# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://github.com/ArnavP2305/flyrank-ml-internship-2/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

### Content Action Triage Framework
Rather than treating model probabilities as abstract numbers, we convert them into clear, human-readable editorial tasks. Each scored page is mapped to a specific action and a trusted reason code to help strategists understand **why** the page was flagged.

#### Triage Rule Logic:
1. **`REFRESH_CONTENT`** (Reason: `STALE_DECAY_RISK`):  
   *Condition:* Model predicted decline probability $\ge 0.60$ and content is stale (`days_since_last_update >= 180`).
2. **`OPTIMIZE_CTR`** (Reason: `CTR_UNDERPERFORMANCE`):  
   *Condition:* Page ranks well (average position $\le 10$) but has low CTR ($< 5.0\%$).
3. **`STABILIZE_POSITION`** (Reason: `STRIKING_DISTANCE_BOOST`):  
   *Condition:* Page is in striking distance ($10 < \text{avg\_position} \le 20$) and carries high volume (impressions $\ge 500$).
4. **`MONITOR`** (Reason: `STABLE_PERFORMANCE`):  
   *Condition:* Pages not meeting decay risk or optimization thresholds.

Below we execute the classification pipeline and print the action distribution across the portfolio.

In [1]:
# Run model prediction, assign actions, and output queue
import os, sys
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
import json

while not os.path.isdir('data/raw') and os.getcwd() != '/':
    os.chdir('..')

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = df['trend_direction'].str.lower().eq('down').astype(int)

# Fit Random Forest model under Grouped Split to get probabilities
features = ['impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'content_age_days', 'word_count']
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df['is_declining'].values
groups = df['client_id'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr, te = next(gss.split(X, y, groups))

rf = RandomForestClassifier(n_estimators=100, max_depth=6, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X.iloc[tr], y[tr])
df['pred_prob'] = rf.predict_proba(X)[:, 1]

# Map archetypes to action and reason codes
actions = []
reasons = []

for idx, row in df.iterrows():
    if row['pred_prob'] >= 0.60 and row['days_since_last_update'] >= 180:
        actions.append('REFRESH_CONTENT')
        reasons.append('STALE_DECAY_RISK')
    elif row['avg_position'] <= 10.0 and row['ctr'] < 0.05:
        actions.append('OPTIMIZE_CTR')
        reasons.append('CTR_UNDERPERFORMANCE')
    elif 10.0 < row['avg_position'] <= 20.0 and row['impressions_90d'] >= 500:
        actions.append('STABILIZE_POSITION')
        reasons.append('STRIKING_DISTANCE_BOOST')
    else:
        actions.append('MONITOR')
        reasons.append('STABLE_PERFORMANCE')

df['action_label'] = actions
df['reason_code'] = reasons

# Print action distribution
print('=== Editorial Queue Action Distribution ===')
print(df['action_label'].value_counts().to_string())
print('\n=== Editorial Queue Reason Distribution ===')
print(df['reason_code'].value_counts().to_string())

=== Editorial Queue Action Distribution ===
action_label
MONITOR               19425
OPTIMIZE_CTR           6066
STABILIZE_POSITION     4454
REFRESH_CONTENT          55

=== Editorial Queue Reason Distribution ===
reason_code
STABLE_PERFORMANCE         19425
CTR_UNDERPERFORMANCE        6066
STRIKING_DISTANCE_BOOST     4454
STALE_DECAY_RISK              55


## 2. Intended use and limits

### Intended Use:
- **Decision-Support Tool:** This playbook acts as a prioritisation matrix for content marketing strategists.   It helps identify which pages are most likely to experience a traffic decline or need optimization first.
- **Non-automated Triaging:** Strategy leaders review the queue weekly to assign content refresh tasks to writers.

### Known Limitations:
1. **No Algorithmic Forecasting:** The model does not predict or replicate Google's internal search algorithm.    It merely identifies historical decay patterns in our observed portfolio.
2. **Data Scope Constrained:** Results are valid for high-volume content pages (impressions $\ge 100$).    Low-volume long-tail pages receive poor predictions due to highly volatile click signals.
3. **No Causal Guarantee:** Refreshing a flagged page does not guarantee ranking recovery;    it simply mitigates one observed risk factor (staleness).

In [2]:
# Demonstrate limitations on low-volume pages
low_vol = df[df['impressions_90d'] < 100]
print(f'Average prediction probability on low-volume pages (<100 imp): {low_vol["pred_prob"].mean():.4f}')
print(f'Decline base rate on low-volume pages:                          {low_vol["is_declining"].mean():.4f}')
print('Observation: Prediction probabilities compress toward the mean on thin data, proving inaccuracy.')

Average prediction probability on low-volume pages (<100 imp): 0.3942
Decline base rate on low-volume pages:                          0.3890
Observation: Prediction probabilities compress toward the mean on thin data, proving inaccuracy.


## 3. Human review + the no-go list

### Human Review Rules (Look Before You Edit)
Every recommended action must pass a 3-step manual verification before a writer touches the CMS:

1. **Intent Fit Verification:** Verify if the target queries have shifted intent (e.g. from informational to transactional).    If intent has shifted, rewriting existing content won't recover rankings; a structural pivot is required.
2. **Competitive Landscape Audit:** Inspect Search Engine Result Pages (SERPs) to see if Google has added new SERP features    (e.g., ads, maps, AI Overviews) at the top. If so, a decline in organic CTR is driven by layout changes, not content quality.
3. **Authority check:** Ensure the page still holds backlinks and internal link equity. If internal paths were broken,    redirect fixes must take priority over content updates.

### The Automation No-Go List
- **NEVER automate content generation/overwrites:** Automated AI content deployment without human proofing causes   brand alignment decay and risks search engine penalty flags for low-quality programmatic spam.
- **NEVER automate page redirects:** Changing URLs programmatically destroys link equity if done incorrectly.   All redirect decisions require direct SEO specialist supervision.

In [3]:
# Showcase top candidate that should NOT be automated
high_risk = df[(df['action_label'] == 'REFRESH_CONTENT') & (df['avg_position'] > 0) & (df['avg_position'] <= 10.0)].head(3)
print('=== High-Ranked Pages Flagged for Refresh (Caution: Do NOT automate) ===')
print(high_risk[['content_id', 'pred_prob', 'avg_position', 'days_since_last_update', 'action_label']].to_string(index=False))
print('\nWhy caution is needed: These pages rank in the top-10. Automated updates risk disrupting ')
print('their stable ranking signals.')

=== High-Ranked Pages Flagged for Refresh (Caution: Do NOT automate) ===
          content_id  pred_prob  avg_position  days_since_last_update    action_label
content_fd16e3475c29   0.700382           9.0                     183 REFRESH_CONTENT
content_164eee6bf9c1   0.661253           5.8                     183 REFRESH_CONTENT
content_6557f2b648e8   0.659094           9.2                     183 REFRESH_CONTENT

Why caution is needed: These pages rank in the top-10. Automated updates risk disrupting 
their stable ranking signals.


## 4. Monitoring / retrain triggers

### Triggers to Retrain the Model:
1. **Data Drift (Baseline Shift):** If the average position of the top 100 pages shifts by $\ge 3$ ranks    portfolio-wide, the underlying features have drifted. Retrain immediately.
2. **Layout Disruption:** If Google rolls out a Core Update or a major layout update (like widespread AI Overviews),    CTR profiles change permanently. Retrain the model using post-update performance data.
3. **Standard Schedule:** Re-fit features and train the model every **90 days** to integrate new crawl logs    and freshness indicators.

In [4]:
# Print out monitoring KPIs to watch
df_valid_pos = df[df['avg_position'] > 0]
print('=== Current Portfolio Baseline KPIs ===')
print(f'Average Position (Top 100 pages): {df_valid_pos.nsmallest(100, "avg_position")["avg_position"].mean():.2f}')
print(f'Average CTR (Top 100 pages):      {df_valid_pos.nsmallest(100, "avg_position")["ctr"].mean():.4f}')
print(f'Average Days Since Last Update:    {df["days_since_last_update"].mean():.1f} days')

=== Current Portfolio Baseline KPIs ===
Average Position (Top 100 pages): 0.60
Average CTR (Top 100 pages):      6.3700
Average Days Since Last Update:    46.1 days


## 5. Exports for the paper

We export the ranked triage queue to `work/outputs/actionable_refresh_queue.csv` and create a visualization of the predicted decline probabilities to include in the deployed research report next week.

In [5]:
# Export queue, metrics JSON, and generate decline probability distribution chart
import matplotlib.pyplot as plt

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# 1. Save queue (ignored by git via leak-guard)
df_queue = df.sort_values(by='pred_prob', ascending=False)
df_queue[['client_id', 'content_id', 'pred_prob', 'action_label', 'reason_code']].to_csv(
    'work/outputs/actionable_refresh_queue.csv', index=False
)
print('Wrote triage queue to work/outputs/actionable_refresh_queue.csv')

# 2. Save metrics JSON
metrics = {
    'total_analyzed_pages': int(df.shape[0]),
    'refresh_content_count': int((df['action_label'] == 'REFRESH_CONTENT').sum()),
    'optimize_ctr_count': int((df['action_label'] == 'OPTIMIZE_CTR').sum()),
    'stabilize_position_count': int((df['action_label'] == 'STABILIZE_POSITION').sum()),
    'monitor_count': int((df['action_label'] == 'MONITOR').sum())
}
with open('work/outputs/playbook_summary.json', 'w') as f:
    json.dump(metrics, f)
print('Wrote metrics receipt to work/outputs/playbook_summary.json')

# 3. Plot decline probability distribution
plt.figure(figsize=(8, 5))
plt.hist(df['pred_prob'], bins=30, color='#6C5B7B', edgecolor='white', alpha=0.9)
plt.title('Distribution of Predicted Decline Probabilities', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Predicted Decline Probability', fontsize=10, labelpad=10)
plt.ylabel('Number of Pages', fontsize=10, labelpad=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('work/figures/decline_probability_distribution.png', dpi=300)
plt.close()
print('Generated and saved figure to work/figures/decline_probability_distribution.png')

Wrote triage queue to work/outputs/actionable_refresh_queue.csv
Wrote metrics receipt to work/outputs/playbook_summary.json


Generated and saved figure to work/figures/decline_probability_distribution.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.